# Fixed preprocessing for MovieLens 1M implicit-feedback experiments

This notebook prepares a strict weak-generalization split for EASE experiments: positive implicit feedback, per-user train/validation/test split, common user-item coordinate system, no user-item leakage, and explicit reporting of cold-item drops.

In [1]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
from scipy import sparse

try:
    import requests
except Exception:
    requests = None

SEED = 52
RNG = np.random.default_rng(SEED)

# Use relative project paths, not an absolute user-specific path.
ROOT = Path.cwd()
DATA_ROOT = ROOT / "data" / "raw" / "movielens_1m"
OUT_DIR = ROOT / "data" / "processed" / "ml1m_implicit_random_u3_i5_seed52"
OUT_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
ZIP_PATH = DATA_ROOT / "ml-1m.zip"
EXTRACTED_DIR = DATA_ROOT / "ml-1m"

MIN_RATING = 4
MIN_USER_POSITIVES = 3
MIN_ITEM_POSITIVES = 5
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SPLIT_MODE = "random_by_user"  # for this thesis: static weak generalization, not temporal forecasting
DROP_COLD_EVAL_ITEMS = True

print("OUT_DIR:", OUT_DIR)

OUT_DIR: /Users/alapofis/ease-optimization/notebooks/data/processed/ml1m_implicit_random_u3_i5_seed52


In [2]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    if requests is None:
        raise RuntimeError("requests is not installed and ml-1m.zip is not present locally")
    with requests.get(URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("Downloaded:", ZIP_PATH)

if not (EXTRACTED_DIR / "ratings.dat").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Extracted to:", DATA_ROOT)

ratings_path = EXTRACTED_DIR / "ratings.dat"
movies_path = EXTRACTED_DIR / "movies.dat"
users_path = EXTRACTED_DIR / "users.dat"

ratings = pd.read_csv(
    ratings_path,
    sep="::",
    engine="python",
    header=None,
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin1",
)

movies = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    header=None,
    names=["movie_id", "title", "genres"],
    encoding="latin1",
)

print(ratings.head())
print("ratings:", ratings.shape)

Downloaded: /Users/alapofis/ease-optimization/notebooks/data/raw/movielens_1m/ml-1m.zip
Extracted to: /Users/alapofis/ease-optimization/notebooks/data/raw/movielens_1m
   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291
ratings: (1000209, 4)


In [3]:
# Convert explicit MovieLens ratings to positive implicit feedback.
# Ratings below MIN_RATING are not negative examples; they are treated as unobserved.
pos = ratings.loc[ratings["rating"] >= MIN_RATING, ["user_id", "movie_id", "timestamp"]].copy()
pos["y"] = 1

# MovieLens 1M normally has no duplicate user-item ratings, but keep the guard.
pos = (
    pos.sort_values("timestamp")
       .drop_duplicates(["user_id", "movie_id"], keep="last")
       .reset_index(drop=True)
)

# Iterative k-core filtering: enough positives per user for train/validation/test,
# and enough positives per item to reduce cold-item evaluation artifacts.
def apply_kcore(df: pd.DataFrame, min_user: int, min_item: int) -> pd.DataFrame:
    out = df.copy()
    while True:
        before = len(out)
        user_counts = out.groupby("user_id").size()
        good_users = user_counts[user_counts >= min_user].index
        out = out[out["user_id"].isin(good_users)]

        item_counts = out.groupby("movie_id").size()
        good_items = item_counts[item_counts >= min_item].index
        out = out[out["movie_id"].isin(good_items)]

        if len(out) == before:
            break
    return out.reset_index(drop=True)

pos = apply_kcore(pos, MIN_USER_POSITIVES, MIN_ITEM_POSITIVES)

print("positive implicit interactions after filtering:", len(pos))
print("users:", pos.user_id.nunique())
print("items:", pos.movie_id.nunique())
print("min positives/user:", pos.groupby("user_id").size().min())
print("min positives/item:", pos.groupby("movie_id").size().min())

positive implicit interactions after filtering: 574380
users: 6035
items: 3125
min positives/user: 4
min positives/item: 5


In [4]:
def split_one_user(g: pd.DataFrame, rng: np.random.Generator) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(g)
    if n < 3:
        raise ValueError("Each user must have at least 3 positives before splitting")

    if SPLIT_MODE == "temporal_by_user":
        g2 = g.sort_values("timestamp")
        test = g2.iloc[[-1]]
        valid = g2.iloc[[-2]]
        train = g2.iloc[:-2]
    elif SPLIT_MODE == "random_by_user":
        perm = rng.permutation(n)
        n_test = max(1, int(round(TEST_RATIO * n)))
        n_valid = max(1, int(round(VAL_RATIO * n)))
        if n_test + n_valid >= n:
            n_test = 1
            n_valid = 1
        test_idx = perm[:n_test]
        valid_idx = perm[n_test:n_test + n_valid]
        train_idx = perm[n_test + n_valid:]
        train = g.iloc[train_idx]
        valid = g.iloc[valid_idx]
        test = g.iloc[test_idx]
    else:
        raise ValueError(f"Unknown SPLIT_MODE={SPLIT_MODE}")

    return train, valid, test

train_parts, valid_parts, test_parts = [], [], []
for _, g in pos.groupby("user_id", sort=False):
    tr, va, te = split_one_user(g, RNG)
    train_parts.append(tr)
    valid_parts.append(va)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
valid_df = pd.concat(valid_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

# For a pure collaborative model, items unseen in train cannot be learned meaningfully.
# Drop such validation/test interactions and report how many were removed.
train_items = set(train_df.movie_id.unique())
valid_before = len(valid_df)
test_before = len(test_df)
if DROP_COLD_EVAL_ITEMS:
    valid_df = valid_df[valid_df.movie_id.isin(train_items)].copy()
    test_df = test_df[test_df.movie_id.isin(train_items)].copy()

cold_valid_dropped = valid_before - len(valid_df)
cold_test_dropped = test_before - len(test_df)

print("train/valid/test interactions:", len(train_df), len(valid_df), len(test_df))
print("cold valid interactions dropped:", cold_valid_dropped)
print("cold test interactions dropped:", cold_test_dropped)
print("valid users with positives:", valid_df.user_id.nunique())
print("test users with positives:", test_df.user_id.nunique())

train/valid/test interactions: 459438 57471 57471
cold valid interactions dropped: 0
cold test interactions dropped: 0
valid users with positives: 6035
test users with positives: 6035


In [5]:
# Use a common coordinate system. Users are taken from train; every kept user has train history.
# Items are taken from train so the model catalogue equals the learnable catalogue.
user_ids = np.sort(train_df.user_id.unique())
item_ids = np.sort(train_df.movie_id.unique())

user2idx = {int(u): i for i, u in enumerate(user_ids)}
item2idx = {int(m): j for j, m in enumerate(item_ids)}

# Filter evaluation to the same coordinate system.
valid_df = valid_df[valid_df.user_id.isin(user2idx) & valid_df.movie_id.isin(item2idx)].copy()
test_df = test_df[test_df.user_id.isin(user2idx) & test_df.movie_id.isin(item2idx)].copy()


def to_csr(df: pd.DataFrame, n_users: int, n_items: int) -> sparse.csr_matrix:
    rows = df.user_id.map(user2idx).to_numpy(dtype=np.int64)
    cols = df.movie_id.map(item2idx).to_numpy(dtype=np.int64)
    data = np.ones(len(df), dtype=np.float32)
    X = sparse.csr_matrix((data, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)
    X.sum_duplicates()
    X.data[:] = 1.0
    X.eliminate_zeros()
    return X

X_train = to_csr(train_df, len(user_ids), len(item_ids))
X_valid = to_csr(valid_df, len(user_ids), len(item_ids))
X_test = to_csr(test_df, len(user_ids), len(item_ids))

print("X_train:", X_train.shape, X_train.nnz)
print("X_valid:", X_valid.shape, X_valid.nnz)
print("X_test:", X_test.shape, X_test.nnz)

X_train: (6035, 3125) 459438
X_valid: (6035, 3125) 57471
X_test: (6035, 3125) 57471


In [6]:
def assert_no_overlap(A: sparse.csr_matrix, B: sparse.csr_matrix, name: str) -> None:
    overlap = A.multiply(B).nnz
    assert overlap == 0, f"Leakage in {name}: {overlap} overlapping user-item pairs"

assert X_train.shape == X_valid.shape == X_test.shape
assert_no_overlap(X_train, X_valid, "train-validation")
assert_no_overlap(X_train, X_test, "train-test")
assert_no_overlap(X_valid, X_test, "validation-test")
assert np.all(np.isin(X_train.data, [1.0]))
assert np.all(np.isin(X_valid.data, [1.0]))
assert np.all(np.isin(X_test.data, [1.0]))
assert (np.diff(X_train.indptr) > 0).all(), "Every user must have train history"

stats = pd.DataFrame([
    {"split": "train", "n_users": X_train.shape[0], "n_items": X_train.shape[1], "nnz": X_train.nnz, "density": X_train.nnz / (X_train.shape[0] * X_train.shape[1])},
    {"split": "validation", "n_users": X_valid.shape[0], "n_items": X_valid.shape[1], "nnz": X_valid.nnz, "density": X_valid.nnz / (X_valid.shape[0] * X_valid.shape[1])},
    {"split": "test", "n_users": X_test.shape[0], "n_items": X_test.shape[1], "nnz": X_test.nnz, "density": X_test.nnz / (X_test.shape[0] * X_test.shape[1])},
])

audit = {
    "dataset": "MovieLens 1M",
    "seed": SEED,
    "min_rating_positive": MIN_RATING,
    "min_user_positives": MIN_USER_POSITIVES,
    "min_item_positives": MIN_ITEM_POSITIVES,
    "split_mode": SPLIT_MODE,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "drop_cold_eval_items": DROP_COLD_EVAL_ITEMS,
    "cold_valid_interactions_dropped": int(cold_valid_dropped),
    "cold_test_interactions_dropped": int(cold_test_dropped),
    "n_users": int(X_train.shape[0]),
    "n_items": int(X_train.shape[1]),
    "train_nnz": int(X_train.nnz),
    "valid_nnz": int(X_valid.nnz),
    "test_nnz": int(X_test.nnz),
}

print(json.dumps(audit, ensure_ascii=False, indent=2))
display(stats)

{
  "dataset": "MovieLens 1M",
  "seed": 52,
  "min_rating_positive": 4,
  "min_user_positives": 3,
  "min_item_positives": 5,
  "split_mode": "random_by_user",
  "val_ratio": 0.1,
  "test_ratio": 0.1,
  "drop_cold_eval_items": true,
  "cold_valid_interactions_dropped": 0,
  "cold_test_interactions_dropped": 0,
  "n_users": 6035,
  "n_items": 3125,
  "train_nnz": 459438,
  "valid_nnz": 57471,
  "test_nnz": 57471
}


,split,n_users,n_items,nnz,density
0,train,6035,3125,459438,0.024361
1,validation,6035,3125,57471,0.003047
2,test,6035,3125,57471,0.003047


In [7]:
sparse.save_npz(OUT_DIR / "train.npz", X_train)
sparse.save_npz(OUT_DIR / "valid.npz", X_valid)
sparse.save_npz(OUT_DIR / "test.npz", X_test)

train_df.to_csv(OUT_DIR / "train_interactions.csv", index=False)
valid_df.to_csv(OUT_DIR / "valid_interactions.csv", index=False)
test_df.to_csv(OUT_DIR / "test_interactions.csv", index=False)

pd.DataFrame({"user_id": user_ids, "user_idx": np.arange(len(user_ids))}).to_csv(OUT_DIR / "user_mapping.csv", index=False)
pd.DataFrame({"movie_id": item_ids, "item_idx": np.arange(len(item_ids))}).to_csv(OUT_DIR / "item_mapping.csv", index=False)
stats.to_csv(OUT_DIR / "split_stats.csv", index=False)

with open(OUT_DIR / "preprocessing_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

print("Saved processed split to:", OUT_DIR)
print("Use these files in your experiment config if load_split supports npz triplets, or adapt load_split to read train.npz/valid.npz/test.npz.")

Saved processed split to: /Users/alapofis/ease-optimization/notebooks/data/processed/ml1m_implicit_random_u3_i5_seed52
Use these files in your experiment config if load_split supports npz triplets, or adapt load_split to read train.npz/valid.npz/test.npz.
